# Datenbasis für die Metrik-Auswertung

Erzeugt die In-Memory-SQLite-Datenbank und die Beispielobjekte für `metrics_evaluation.ipynb`. Dort wird dieses Notebook mit `%run ./metrics_data_setup.ipynb` eingebunden.

Das ORM-Modul `krrood_test/dataset/ormatic_interface.py` entsteht erst, wenn pytest die krrood-Tests einsammelt. Deshalb wird vor dem Import einmal `pytest --collect-only` in einem eigenen Prozess gestartet.

In [1]:
import subprocess
import sys

TEST_ROOT = "/home/jovyan/krrood/test"

for path in (TEST_ROOT, "/home/jovyan/krrood"):
    if path not in sys.path:
        sys.path.insert(0, path)

# Collecting the tests generates dataset/ormatic_interface.py.
result = subprocess.run(
    [sys.executable, "-m", "pytest", "krrood_test/test_ormatic/test_eql.py",
     "-q", "--collect-only"],
    cwd=TEST_ROOT, capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stdout[-3000:])
    print(result.stderr[-3000:])
    raise RuntimeError("Generating ormatic_interface.py failed, see output above.")

from sqlalchemy.orm import sessionmaker, configure_mappers

from krrood.ormatic.utils import create_engine
from test.krrood_test.dataset.ormatic_interface import Base, BodyDAO
from test.krrood_test.dataset.semantic_world_like_classes import (
    Container,
    Handle,
    FixedConnection,
    PrismaticConnection,
    World,
)


def create_demo_session():
    """
    Create a fresh in-memory SQLite session, set up like the engine/session
    fixtures in krrood_test/conftest.py. Every call starts with an empty
    database.
    """
    configure_mappers()
    engine = create_engine("sqlite:///:memory:")
    Base.metadata.create_all(engine)
    return sessionmaker(bind=engine)()


def build_bodies_for_simple_filter(session, sizes=(1, 3, 6, 8, 10)):
    """
    Insert bodies with the given sizes, on both sides of the threshold 5
    used by the filter queries.

    :return: Number of inserted rows
    """
    for i, size in enumerate(sizes, start=1):
        session.add(BodyDAO(name=f"Body{i}", size=size))
    session.commit()
    return len(sizes)


def build_container_handle_world():
    """World with one container, one handle and a fixed connection (3 variables)."""
    world = World(1, [Container("C1"), Handle("H1")])
    world.connections = [FixedConnection(world.bodies[0], world.bodies[1])]
    return world


def build_four_variable_world():
    """World with two containers, two handles, a fixed and a prismatic connection (4 variables)."""
    world = World(1, [Container("C1"), Container("C2"), Handle("H1"), Handle("H2")])
    world.connections = [
        FixedConnection(world.bodies[0], world.bodies[2]),
        PrismaticConnection(world.bodies[1], world.bodies[0]),
    ]
    return world


def build_container_chain(k: int):
    """
    Chain of k containers connected by k - 1 fixed connections:
    C0 --FC0-- C1 --FC1-- ... -- C(k-1). Used for the scaling series.
    """
    bodies = [Container(f"C{i}") for i in range(k)]
    world = World(1, bodies)
    world.connections = [FixedConnection(bodies[i], bodies[i + 1]) for i in range(k - 1)]
    return world